# <font color="#418FDE" size="6.5" uppercase>**Responsible Deployment**</font>

>Last update: 20260708.
    
By the end of this Lecture, you will be able to:
- Evaluate competing models using metrics, plots, interpretability, and engineering consequences. 
- Identify risks involving bias, imbalance, explainability, data drift, and unsafe automation. 
- Design a responsible final machine learning workflow for a civil engineering use case. 


## **1. Responsible Model Selection**

### **1.1. Model Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_01_01.jpg?v=1783569540" width="250">



>* Treat metrics as evidence, not answers
>* Choose metrics based on engineering consequences

>* Check performance across sites and conditions
>* Use diagnostics to reveal errors and calibration

>* Choose models based on engineering consequences
>* Use stakeholder input for safe deployment



In [ ]:
#@title Python Code - Model Metrics

# Compare metrics for responsible model selection.
# Civil engineering errors have different consequences.
# Small examples keep the lesson transparent.

import numpy as np
import matplotlib.pyplot as plt

# Store true bridge defect labels.
y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])

# Store predictions from two candidate models.
model_a = np.array([1, 0, 0, 1, 0, 0, 0, 0, 1, 0])
model_b = np.array([1, 1, 1, 0, 1, 1, 0, 0, 0, 0])

# Validate equal vector lengths.
assert y_true.size == model_a.size == model_b.size

# Define a compact metric function.
def metrics(y, pred):
    tp = int(np.sum((y == 1) & (pred == 1)))
    tn = int(np.sum((y == 0) & (pred == 0)))
    fp = int(np.sum((y == 0) & (pred == 1)))

    fn = int(np.sum((y == 1) & (pred == 0)))
    accuracy = (tp + tn) / y.size
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    return accuracy, precision, recall, fp, fn

# Compute metrics for both models.
a = metrics(y_true, model_a)
b = metrics(y_true, model_b)

# Print concise engineering interpretation.
print("Model A accuracy, precision, recall:", np.round(a[:3], 2))
print("Model B accuracy, precision, recall:", np.round(b[:3], 2))
print("Model A false alarms, missed defects:", a[3], a[4])
print("Model B false alarms, missed defects:", b[3], b[4])

# Plot metrics for visual comparison.
labels = ["Accuracy", "Precision", "Recall"]
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width / 2, a[:3], width, label="Model A")
ax.bar(x + width / 2, b[:3], width, label="Model B")

ax.set_ylim(0, 1.05)
ax.set_ylabel("Metric value")
ax.set_title("Bridge defect screening metrics")
ax.set_xticks(x, labels)

ax.legend()
plt.tight_layout()
plt.show()



### **1.2. Interpretability Insights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_01_02.jpg?v=1783569538" width="250">



>* Look beyond accuracy to model reasoning
>* Prefer explanations that support engineering judgment

>* Global explanations reveal overall model behavior
>* Local explanations support case-level engineering checks

>* Treat interpretability as deployment evidence
>* Match explanations to users and accountability



In [ ]:
#@title Python Code - Interpretability Insights

# Interpretability supports responsible engineering model selection.
# Simple models can reveal influential variables.
# Explanations should match civil engineering judgment.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny bridge deterioration dataset.
np.random.seed(7)
n = 40
age_years = np.random.randint(5, 80, n)

# Add engineering features with plausible relationships.
crack_mm = np.random.uniform(0, 12, n)
traffic_k = np.random.uniform(2, 55, n)
inspection_month = np.random.randint(1, 13, n)

# Build a risk score from physical indicators.
risk = 0.04 * age_years + 0.55 * crack_mm
risk = risk + 0.03 * traffic_k
risk = risk + np.random.normal(0, 0.8, n)

# Store features in a small table.
data = pd.DataFrame({
    "age_years": age_years,
    "crack_mm": crack_mm,
    "traffic_k": traffic_k,
    "inspection_month": inspection_month,
    "risk_score": risk,
})

# Define two competing linear explanations.
physical_features = ["age_years", "crack_mm", "traffic_k"]
proxy_features = ["inspection_month", "traffic_k", "age_years"]

# Fit least squares models using NumPy.
def fit_linear(frame, features, target):
    X = frame[features].to_numpy(dtype=float)
    y = frame[target].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return beta

# Compute predictions and mean absolute error.
def predict_linear(frame, features, beta):
    X = frame[features].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(X)), X])
    return X @ beta

# Fit both candidate models.
beta_physical = fit_linear(data, physical_features, "risk_score")
beta_proxy = fit_linear(data, proxy_features, "risk_score")
pred_physical = predict_linear(data, physical_features, beta_physical)

# Evaluate both models with one metric.
pred_proxy = predict_linear(data, proxy_features, beta_proxy)
mae_physical = np.mean(np.abs(data["risk_score"] - pred_physical))
mae_proxy = np.mean(np.abs(data["risk_score"] - pred_proxy))

# Convert coefficients into comparable importance values.
def importance_table(features, beta, frame):
    scales = frame[features].std().to_numpy(dtype=float)
    values = np.abs(beta[1:] * scales)
    return pd.Series(values, index=features).sort_values()

# Prepare global interpretability summaries.
imp_physical = importance_table(physical_features, beta_physical, data)
imp_proxy = importance_table(proxy_features, beta_proxy, data)

# Print concise responsible selection evidence.
print("Responsible model selection example")
print(f"Physical-feature MAE: {mae_physical:.2f}")
print(f"Proxy-feature MAE: {mae_proxy:.2f}")
print("Top physical-model driver:", imp_physical.index[-1])
print("Top proxy-model driver:", imp_proxy.index[-1])
print("Engineering check: prefer explanations tied to deterioration mechanisms.")

# Plot global importance for both models.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
imp_physical.plot(kind="barh", ax=axes[0], color="steelblue")
imp_proxy.plot(kind="barh", ax=axes[1], color="darkorange")

# Label plots for interpretation.
axes[0].set_title("Physical features")
axes[1].set_title("Proxy-risk features")
axes[0].set_xlabel("Standardized importance")
axes[1].set_xlabel("Standardized importance")

# Display one compact interpretability plot.
plt.tight_layout()
plt.show()



### **1.3. Operational Consequences**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_01_03.jpg?v=1783569541" width="250">



>* Consider real-world impacts of model errors
>* Choose models aligned with engineering priorities

>* Consider resources needed for long-term model use
>* Choose models fitting real field workflows

>* Model outputs shape decisions and accountability
>* Use human review and ongoing monitoring



In [ ]:
#@title Python Code - Operational Consequences

# Compare models using operational consequences.
# Bridge warnings have asymmetric field costs.
# Metrics alone can hide deployment risk.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Store confusion counts for two models.
models = pd.DataFrame({
    "model": ["Cautious", "Aggressive"],
    "false_alarms": [18, 6],
    "missed_warnings": [2, 9],
})

# Define simple operational cost assumptions.
inspection_cost = 2500
missed_warning_cost = 120000
models["total_cost"] = (
    models["false_alarms"] * inspection_cost
    + models["missed_warnings"] * missed_warning_cost
)

# Add a familiar accuracy-style metric.
total_cases = 100
models["accuracy"] = (
    total_cases
    - models["false_alarms"]
    - models["missed_warnings"]
) / total_cases

# Validate the tiny teaching table.
assert len(models) == 2
assert models["total_cost"].min() > 0

# Print a compact operational comparison.
print("Responsible model selection example")
for row in models.itertuples(index=False):
    print(f"{row.model}: accuracy={row.accuracy:.0%}, cost=${row.total_cost:,.0f}")

# Identify the lower consequence option.
best = models.loc[models["total_cost"].idxmin(), "model"]
print(f"Lower operational consequence: {best}")

# Plot cost components for field interpretation.
plot_data = models.set_index("model")[["false_alarms", "missed_warnings"]]
ax = plot_data.plot(kind="bar", color=["steelblue", "tomato"])

# Label the plot for engineering discussion.
ax.set_title("Error types behind operational consequences")
ax.set_ylabel("Number of cases")
ax.set_xlabel("Candidate model")
plt.xticks(rotation=0)

# Keep the single plot readable.
plt.tight_layout()
plt.show()



## **2. Risk and Ethics**

### **2.1. Data Drift Risks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_02_01.jpg?v=1783569536" width="250">



>* Models weaken when real-world data changes
>* Unmonitored drift can make decisions unsafe

>* Drift changes inputs, targets, or relationships
>* Confident outputs can hide outdated predictions

>* Monitor drift throughout the model lifecycle
>* Plan retraining, accountability, and human oversight



In [ ]:
#@title Python Code - Data Drift Risks

# Data drift can quietly reduce model reliability.
# Civil engineers must monitor changing operating conditions.
# This example compares baseline and deployed sensor data.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set a seed for repeatable teaching results.
rng = np.random.default_rng(42)

# Create baseline bridge vibration readings in millimeters.
baseline = rng.normal(loc=2.0, scale=0.25, size=120)

# Create deployed readings after heavier traffic appears.
deployed = rng.normal(loc=2.35, scale=0.35, size=120)

# Validate both samples before comparing distributions.
assert baseline.size == deployed.size == 120

# Summarize each period with simple engineering statistics.
summary = pd.DataFrame({
    "period": ["development", "deployment"],
    "mean_mm": [baseline.mean(), deployed.mean()],
    "std_mm": [baseline.std(), deployed.std()],
})

# Compute a simple drift signal using mean shift.
pooled_std = np.sqrt((baseline.var() + deployed.var()) / 2)

# Avoid division problems if readings are constant.
mean_shift = abs(deployed.mean() - baseline.mean()) / pooled_std

# Choose a transparent threshold for classroom discussion.
drift_flag = mean_shift > 0.8

# Print concise results for responsible deployment review.
print("Data drift check for bridge vibration monitoring")
print(summary.round(3).to_string(index=False))
print(f"Standardized mean shift: {mean_shift:.2f}")
print(f"Drift flag raised: {drift_flag}")
print("Action: review model performance before automated decisions.")

# Plot both distributions to make drift visible.
plt.figure(figsize=(7, 4))

# Use histograms to compare development and deployment data.
plt.hist(baseline, bins=14, alpha=0.65, label="development")
plt.hist(deployed, bins=14, alpha=0.65, label="deployment")

# Label the plot for engineering interpretation.
plt.xlabel("Bridge vibration amplitude, millimeters")
plt.ylabel("Number of readings")
plt.title("Data drift risk after deployment")

# Add a legend and display the single plot.
plt.legend()
plt.tight_layout()
plt.show()



### **2.2. Bias and Imbalance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_02_02.jpg?v=1783569534" width="250">



>* Overall accuracy can hide unfair subgroup errors
>* Check representation and unequal impacts before deployment

>* Rare events can hide behind high accuracy
>* Check performance across important subgroups

>* Document data sources and test subgroup errors
>* Limit deployment until fairness and reliability improve



In [ ]:
#@title Python Code - Bias and Imbalance

# This example shows hidden subgroup risk.
# Overall accuracy can hide unfair errors.
# Civil models need subgroup checks.

# Import small analysis and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic synthetic inspection records.
rng = np.random.default_rng(42)
urban_count, rural_count = 80, 20

# Store subgroup labels for road inspections.
groups = np.array(["Urban"] * urban_count + ["Rural"] * rural_count)
actual = np.array([1] * 30 + [0] * 50 + [1] * 12 + [0] * 8)

# Simulate biased model predictions by subgroup.
urban_pred = np.array([1] * 24 + [0] * 6 + [1] * 5 + [0] * 45)
rural_pred = np.array([1] * 4 + [0] * 8 + [1] * 1 + [0] * 7)

# Combine predictions into one vector.
predicted = np.concatenate([urban_pred, rural_pred])
assert len(groups) == len(actual) == len(predicted)

# Build a compact table for evaluation.
data = pd.DataFrame({"group": groups, "actual": actual, "predicted": predicted})
data["correct"] = data["actual"] == data["predicted"]

# Define a safe recall calculation.
def recall_for_defects(frame):
    positives = frame[frame["actual"] == 1]
    if len(positives) == 0:
        return np.nan
    return (positives["predicted"] == 1).mean()

# Summarize overall and subgroup performance.
overall_accuracy = data["correct"].mean()
summary = data.groupby("group").apply(recall_for_defects, include_groups=False)

# Print only key teaching results.
print(f"Overall accuracy: {overall_accuracy:.0%}")
print("Defect recall by subgroup:")
for group_name, value in summary.items():
    print(f"  {group_name}: {value:.0%}")

# Plot subgroup recall to reveal imbalance.
fig, ax = plt.subplots(figsize=(6, 4))
summary.plot(kind="bar", ax=ax, color=["steelblue", "darkorange"])

# Add responsible deployment reference line.
ax.axhline(overall_accuracy, color="black", linestyle="--", label="overall accuracy")
ax.set_ylim(0, 1)
ax.set_ylabel("Recall for actual defects")

# Label the ethical engineering message.
ax.set_title("Bias risk: rural defects are missed more often")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()



### **2.3. Unsafe Automation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_02_03.jpg?v=1783569532" width="250">



>* Automation needs safeguards and human oversight
>* High-stakes errors can harm safety and equity

>* Overtrusting models can override expert judgment
>* Outputs may miss real-world decision context

>* Use automation to support accountable human decisions
>* Require review, escalation, transparency, and appeals



In [ ]:
#@title Python Code - Unsafe Automation

# Unsafe automation needs human oversight.
# Simple rules can hide uncertainty.
# Engineers must define escalation safeguards.

import math
import statistics

# Create small bridge inspection examples.
bridges = [
    {"name": "A", "score": 0.18, "age": 22, "cracks_in": 0.05},
    {"name": "B", "score": 0.31, "age": 48, "cracks_in": 0.20},
    {"name": "C", "score": 0.44, "age": 65, "cracks_in": 0.35},
    {"name": "D", "score": 0.52, "age": 18, "cracks_in": 0.02},
]

# Validate the tiny teaching dataset.
assert len(bridges) == 4
assert all(0 <= item["score"] <= 1 for item in bridges)

# Define automation and escalation thresholds.
auto_close_limit = 0.30
human_review_limit = 0.50

# Convert model scores into deployment actions.
actions = []
for item in bridges:
    if item["score"] < auto_close_limit:
        action = "auto-close"
    elif item["score"] < human_review_limit:
        action = "engineer review"
    else:
        action = "urgent inspection"
    actions.append(action)

# Add context that automation might miss.
context_flags = []
for item in bridges:
    old_bridge = item["age"] > 50
    visible_crack = item["cracks_in"] > 0.25
    context_flags.append(old_bridge or visible_crack)

# Count unsafe automation cases.
unsafe_cases = 0
for action, flag in zip(actions, context_flags):
    unsafe_cases += int(action == "auto-close" and flag)

# Summarize the deployment lesson.
print("Unsafe automation demo: bridge inspection triage")
print(f"Automated closures: {actions.count('auto-close')}")
print(f"Human reviews required: {actions.count('engineer review')}")
print(f"Urgent inspections: {actions.count('urgent inspection')}")
print(f"Unsafe auto-close cases: {unsafe_cases}")
print("Safeguard: escalate uncertain or context-conflicting cases.")



## **3. Responsible Final Workflow**

### **3.1. Workflow Planning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_03_01.jpg?v=1783569543" width="250">



>* Define the decision and model limits
>* Plan uncertainty, escalation, and no-use conditions

>* Validate diverse data before model use
>* Trace decisions and communicate uncertainty clearly

>* Plan monitoring, updates, and user responsibilities
>* Document decisions for transparent, accountable audits



In [ ]:
#@title Python Code - Workflow Planning

# Plan responsible civil engineering model workflows.
# Connect predictions with checks and handoffs.
# Keep automation bounded by human judgment.

import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny bridge prioritization workflow table.
records = [
    {"bridge": "A", "risk": 0.82, "data_ok": True, "drift": False},
    {"bridge": "B", "risk": 0.61, "data_ok": False, "drift": False},
    {"bridge": "C", "risk": 0.91, "data_ok": True, "drift": True},
    {"bridge": "D", "risk": 0.44, "data_ok": True, "drift": False},
]

# Convert records into a small dataframe.
df = pd.DataFrame(records)
assert len(df) == 4 and "risk" in df.columns

# Define workflow rules before using predictions.
def workflow_action(row):
    if not row["data_ok"]:
        return "hold: verify data"
    if row["drift"]:
        return "escalate: engineer review"
    if row["risk"] >= 0.75:
        return "schedule inspection"
    return "monitor normally"

# Apply the responsible workflow decision rules.
df["action"] = df.apply(workflow_action, axis=1)

# Summarize decisions without printing full dataframes.
print("Responsible workflow example: bridge inspection ranking")
print(f"Records checked: {len(df)}")
print(f"Data holds: {(df['action'].str.contains('hold')).sum()}")
print(f"Engineer escalations: {(df['action'].str.contains('escalate')).sum()}")
print(f"Direct inspections: {(df['action'] == 'schedule inspection').sum()}")

# Plot risk scores with workflow action labels.
colors = ["red" if x >= 0.75 else "steelblue" for x in df["risk"]]
plt.figure(figsize=(7, 4))
plt.bar(df["bridge"], df["risk"], color=colors)
plt.axhline(0.75, color="black", linestyle="--", label="inspection threshold")

# Add concise action labels above bars.
for index, row in df.iterrows():
    plt.text(index, row["risk"] + 0.03, row["action"], ha="center", fontsize=8)

# Format the workflow planning plot.
plt.ylim(0, 1.15)
plt.ylabel("Predicted deterioration risk")
plt.xlabel("Bridge asset")
plt.title("Responsible workflow: prediction plus checks")
plt.legend()
plt.tight_layout()
plt.show()



### **3.2. Ethical Safeguards**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_03_02.jpg?v=1783569545" width="250">



>* Plan safeguards before civil engineering model deployment
>* Check fairness, reliability, and decision limits

>* Check data for bias and missing communities
>* Test subgroups and fix unfair gaps

>* Document limits, data, purpose, and privacy
>* Monitor performance and respond to concerns



In [ ]:
#@title Python Code - Ethical Safeguards

# Ethical safeguards guide responsible engineering deployment.
# Small audits reveal hidden model risks.
# This example checks subgroup safety.

import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny bridge prioritization audit dataset.
data = pd.DataFrame({
    "area": ["urban", "urban", "rural", "rural", "coastal", "coastal"],
    "bridges": [40, 35, 18, 16, 22, 20],
    "missed_defects": [2, 3, 5, 4, 2, 6],
})

# Validate the dataset before calculating rates.
required = {"area", "bridges", "missed_defects"}
assert required.issubset(data.columns), "Missing required audit columns."
assert (data["bridges"] > 0).all(), "Bridge counts must be positive."

# Compute missed defect rates by community context.
data["miss_rate"] = data["missed_defects"] / data["bridges"]
summary = data.groupby("area", as_index=False)["miss_rate"].mean()
summary = summary.sort_values("miss_rate", ascending=False)

# Define a simple ethical safeguard threshold.
threshold = 0.18
flagged = summary[summary["miss_rate"] > threshold]

# Print a compact responsible deployment checklist.
print("Ethical safeguard audit for bridge model.")
print(f"Highest subgroup miss rate: {summary['miss_rate'].max():.1%}.")
print(f"Flagged contexts above threshold: {len(flagged)}.")
print("Action: collect data, add warnings, or limit deployment.")

# Plot subgroup risk to support transparent review.
plt.figure(figsize=(6, 3.5))
plt.bar(summary["area"], summary["miss_rate"], color="steelblue")
plt.axhline(threshold, color="crimson", linestyle="--", label="safeguard threshold")
plt.ylabel("Missed defect rate")
plt.title("Ethical safeguard: subgroup performance audit")
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Human Oversight**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_03_03.jpg?v=1783569546" width="250">



>* Review model outputs before engineering decisions
>* Define evidence, responsibility, and escalation steps

>* Set boundaries for automation and expert review
>* Escalate uncertainty, conflicts, and unfamiliar conditions

>* Record decisions to improve model performance
>* Ensure fair, accountable human-guided outcomes



In [ ]:
#@title Python Code - Human Oversight

# Human oversight keeps model outputs accountable.
# Engineers review uncertain infrastructure predictions before action.
# This example simulates escalation rules.

import pandas as pd
import matplotlib.pyplot as plt

# Create small inspection cases for review.
cases = pd.DataFrame({
    "asset": ["Bridge A", "Culvert B", "Road C", "Wall D"],
    "risk_score": [0.82, 0.41, 0.67, 0.91],

    "confidence": [0.76, 0.88, 0.52, 0.61],
    "field_conflict": [False, False, True, True],
    "critical_asset": [True, False, False, True],
})

# Check the table has expected rows.
assert len(cases.index) == 4, "Expected four inspection cases."

# Define simple oversight thresholds.
high_risk = cases["risk_score"] >= 0.75
low_confidence = cases["confidence"] < 0.65

# Combine technical and engineering triggers.
escalate = high_risk | low_confidence
escalate = escalate | cases["field_conflict"]
escalate = escalate | cases["critical_asset"]

# Store the final oversight decision.
cases["oversight_decision"] = "routine review"
cases.loc[escalate, "oversight_decision"] = "engineer review required"

# Print a compact decision summary.
print("Human oversight example for civil infrastructure.")
print(cases[["asset", "oversight_decision"]].to_string(index=False))
print("Escalation protects safety when risk, uncertainty, or conflict appears.")

# Plot risk and confidence for each asset.
ax = cases.plot(
    x="asset", y=["risk_score", "confidence"], kind="bar"
)

# Add a clear escalation threshold line.
ax.axhline(0.75, color="red", linestyle="--", label="risk threshold")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")

# Label the plot for interpretation.
ax.set_title("Model output plus human oversight triggers")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Responsible Deployment**</font>


In this lecture, you learned to:
- Evaluate competing models using metrics, plots, interpretability, and engineering consequences. 
- Identify risks involving bias, imbalance, explainability, data drift, and unsafe automation. 
- Design a responsible final machine learning workflow for a civil engineering use case. 

<font color='yellow'>Congratulations on completing this course!</font>